In [ ]:
## install required packages
# ! pip install openai gradio pdfplumber python-dotenv geocoder

In [ ]:
# import dependencies
import os
import requests
import json
from openai import OpenAI
import gradio as gr
import pdfplumber as pp
from datetime import date
import geocoder


# load environment variables
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
name = "Martins Awojide"
today = date.today().strftime("%B %d, %Y")
location = geocoder.ip('me').city

In [ ]:
# setup provider for chat completions
client = OpenAI(
                base_url="https://openrouter.ai/api/v1",
                api_key=os.getenv("OPENROUTER_API_KEY")
                )

In [ ]:
mood_model = "openai/gpt-4o-mini-2024-07-18"
# chat_model = "google/gemini-2.5-flash-lite"
chat_model = "gpt-4o-mini-2024-07-18"

In [ ]:
# support for push notifications

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def push_notification(message):
    payload = {
                "user": pushover_user,
                "token": pushover_token,
                "message": message,
                }
    requests.post(pushover_url, data=payload)

In [ ]:
# My resume is the context for my Digital Twin
def get_digital_twin_context(source_document):
    with pp.open(source_document) as pdf:
        context_as_txt = ""
        for page in pdf.pages: context_as_txt += page.extract_text() # convert to TXT
    return context_as_txt

source_document = "resume_mxz.pdf"
context = get_digital_twin_context(source_document)
print(context)

In [ ]:
def system_prompt(name=name, context=context, today=today, location=location):

    return f"""
You are {name}, speaking in the first person and representing {name} in all conversations. \
    You are a digital counterpart — communicating naturally and faithfully, never mechanically.

## Source of Truth
The following context is the only source of truth about your background, \
    including experience, skills, education, and qualifications:

{context}

You are currently in {location} as of {today}. \
    Use this as your reference point when discussing availability or scheduling.

## Grounding Rules
When a user asks about your background, ground your response strictly in the provided context. \
    Do not invent, infer, or extend beyond what is explicitly stated. \
    If a question about your background cannot be answered from context, do not guess — invoke the escalation tool instead (see Tools section). \
    Maintain consistency across your timeline: roles, dates, and achievements must be accurately represented \
    and always associated with the correct position and timeframe.

## Tone and Conversation
Adapt your tone dynamically based on the conversation.

- For greetings, small talk, and general conversation unrelated to your background, respond naturally and conversationally.
- For questions about career, qualifications, hiring, or professional evaluation, bias toward clarity, \
    structure, and credibility — even if the user's tone is informal.
- Never produce harmful, offensive, or extreme content. Avoid any tone that would undermine professional credibility.

Distinguish carefully between casual conversation and requests for specific factual details about {name}. \
    If a user asks for concrete personal information — physical attributes, contact details, clothing sizes, \
    private preferences, or any detail not present in context — this is a factual query, not small talk. \
    Do not guess or deflect with a vague response. Invoke the escalation tool.

## Tools
You have access to two tools. Use them under the exact conditions described below.

**Tool 1 — Schedule Meeting**
Trigger: The user explicitly expresses interest in meeting, speaking, or connecting with {name}.
Action: Before invoking the tool, collect their full name, email address, and the purpose of the meeting. \
    Do not invoke the tool without the email address. Ask for the user's name for more context \
    Once invoked, confirm to the user that the meeting has been scheduled.

**Tool 2 — Escalate Unanswered Question**
Trigger: The user asks for specific personal or factual information about {name} that is not present in context. \
    This includes physical attributes, contact details, private information, unstated preferences, \
    or any concrete personal detail not explicitly covered.
Action: Invoke this tool instead of responding with a generic "I don't know." \
    Do not skip this tool and reply with text alone. \
    Do not invoke the tool without the email address. Ask for the user's name for more context \
    After invoking it, briefly inform the user that their question has been flagged \
    and {name} may follow up directly.

Do not invoke either tool for general small talk, greetings, or questions you can fully answer from context.
"""

In [ ]:
# Infer the mood or tone of the user

def set_user_reply_mood(user_input):

    # infer the mood or tone of the user based on their input
    user_mood_system_prompt = f"You are a mood detector \
                        Your task is to detect the mood or tone of the user based on their input \
                        The mood or tone can be professional, casual, friendly, formal, informal or more \
                        You should only output the user's mood or tone in one or two words."
                        
    mood_response = client.chat.completions.create(
        model = mood_model,
        messages = [
            {"role": "system", "content": user_mood_system_prompt},
            {"role": "user", "content": user_input},
        ]
    )
    mood = mood_response.choices[0].message.content

    # set the mood or tone of the reply based on the user's mood or tone
    reply_mood_system_prompt = f"You are a reply mood setter for a digital twin \
                        Your task is to set the mood or tone of the digital twin's reply based on the user's mood or tone \
                        You should only output the reply mood or tone in one or two words based on the user's mood or tone."
    reply_user_mood_prompt = f"The user's mood or tone is {mood}. What mood or tone should the digital twin's reply be in? \
                        The digital twin should maintain should maintain the mood or tone if it is generally positive \
                        The digital twin should maintain a professional tone if the user's mood or tone is neutral \
                        The digital twin should switch the mood or tone if it is perceived negative, the goal is to descalate, reply with empathy or uplift the user's mood. \
                        You should only output the reply mood or tone in one or two words based on the user's mood or tone."
    
    reply_mood_response = client.chat.completions.create(
        model = mood_model,
        messages = [
            {"role": "system", "content": reply_user_mood_prompt},
            {"role": "user", "content": mood},
        ]
    )
    reply_mood = reply_mood_response.choices[0].message.content

    return reply_mood

In [ ]:
# creating basic tools

# push notification for possible meeting
def notify_with_user_details_for_meeting(email, name="Name not provided", notes="Notes not provided"):
    push_notification(f"{name} with {email} would like to meet! Here are some extra notes:\n\n{notes}")
    return {"notified": "ok"}

# push notification for direct contact on further questions
def notify_on_unknown_question_details(email, question, name="Name not provided"):
    push_notification(f"{name} with {email} would like an answer to this question:\n\n{question}")
    return {"notified": "ok"}

In [ ]:
# creating tools description as JSON objects
# The name of the function is provided to set it up for tool calling

# for setting up meetings 
notify_with_user_details_for_meeting_json = {
    "name": "notify_with_user_details_for_meeting", 
    "description": "Use this tool to record that a user is interested in having a meeting and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

# For answering questions my digital twin doesn't have the answer to
notify_on_unknown_question_details_json = {
    "name": "notify_on_unknown_question_details",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer, if the conversation is going in circles without resolution, or simply longer than 7 back and forths.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
            "email": {
                "type": "string",
                "description": "The email address of the user asking the question"
            },
        },
        "required": ["question", "email"],
        "additionalProperties": False
    }
}

In [ ]:
# aggregating the tools
tools = [
    {"type": "function", "function": notify_with_user_details_for_meeting_json},
    {"type": "function", "function": notify_on_unknown_question_details_json},
]

In [ ]:
# handling tool call(s) in conservations

def handle_tool_calls(tool_calls):

    results = []

    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        tool_arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)

        if tool: result = tool(**tool_arguments)
        else: result = {"error": f"Tool {tool_name} not found"}

        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    
    return results

In [ ]:
# callback function to handle user input and generate response from the digital twin

def chat(message, history):

    reply_mood = set_user_reply_mood(message)
    user_prompt = f"{message}. Reply to this message in a {reply_mood} tone."
    messages = [{"role": "system", "content": system_prompt(name, context, location, today)}] + history + [{"role": "user", "content": user_prompt}]

    this_conversation_done = False

    while not this_conversation_done:

        # call LLM with or without full context and tools' response
        response = client.chat.completions.create(
            model = chat_model,
            messages = messages,
            tools=tools,
        )
        print(f"Response: {response}", flush=True)
        
        # checking if the LLM wants to call a tool and the tool call results as context
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            assistant_response = response.choices[0].message
            tool_calls = assistant_response.tool_calls
            results = handle_tool_calls(tool_calls)
            print(f"{results}", flush=True)
            messages.append(assistant_response)     # add LLM's response to the message list above for final reponse's context
            messages.extend(results)                # add tool(s) response to the message list above for final response augmentation
        else: this_conversation_done = True
    
    # final response with full context and tool call results if there were any
    final_response = response.choices[0].message.content
    if not final_response:
        final_response = "I'm sorry, I could not generate a response."

    return final_response

In [ ]:
# setup the gradio interface for the digital twin

def chat_interface():
    interface = gr.ChatInterface(
                                fn=chat, 
                                type="messages",
                                title=f"Digital Me", 
                                description="You can chat with me anytime here!",
                            )
    interface.launch()

In [ ]:
chat_interface()